<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/%5B03B%5D-CIK_Ticker_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 3b: CIK-to-Ticker Mapping

SEC EDGAR identifies filers by CIK (Central Index Key), not stock ticker.
Yahoo Finance needs a ticker. This pulls the SEC's own free CIK-ticker
mapping file and merges it into your screening worksheet.

**Run this after Part A of Step 3 (worksheet build) and before Step 4
(Yahoo Finance pull).**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 229 (delta 94), reused 169 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (229/229), 1.62 MiB | 11.68 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas requests

## Cell 3 — Configuration

In [ ]:
import os

HEADERS = {"User-Agent": "QM640 Capstone research Shan_muganathan@yahoo.com"}
RAW_DIR = os.path.join(BASE_DIR, "data/raw")

## Cell 4 — Pull the SEC's CIK-ticker crosswalk and merge it in

In [ ]:
import pandas as pd
import requests


def get_cik_ticker_map():
    """SEC's own free CIK<->ticker crosswalk, updated regularly."""
    url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    df = pd.DataFrame.from_dict(data, orient="index")
    df = df.rename(columns={"cik_str": "cik", "title": "company_name"})
    df["cik"] = df["cik"].astype(int)
    out_path = os.path.join(RAW_DIR, "cik_ticker_map.csv")
    df.to_csv(out_path, index=False)
    print(f"CIK-ticker map: {len(df)} entries -> {out_path}")
    return df


def merge_into_screening_worksheet():
    cik_map = get_cik_ticker_map()
    worksheet_path = os.path.join(RAW_DIR, "screening_worksheet.csv")
    worksheet = pd.read_csv(worksheet_path)

    worksheet = worksheet.merge(cik_map[["cik", "ticker"]], on="cik", how="left")
    unmatched = worksheet["ticker"].isna().sum()
    if unmatched:
        print(f"Warning: {unmatched} events have no ticker match - these firms "
              f"may not be on the SEC's mapping (e.g., recently delisted or "
              f"foreign private issuers). Review manually before excluding.")

    worksheet.to_csv(worksheet_path, index=False)
    print(f"Merged ticker into {worksheet_path} ({len(worksheet)} rows)")
    return worksheet


worksheet = merge_into_screening_worksheet()
worksheet.head()

CIK-ticker map: 10432 entries -> /content/QM640-WALSH-CAPSTONE/data/raw/cik_ticker_map.csv
Merged ticker into /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv (9631 rows)


,query,cik,company_name,form_type,file_date,accession_no,adsh,file_name,is_genuine_ai_event,announcement_type,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes,ticker
0,"""AI-powered""",1013857,PEGASYSTEMS INC (PEGA) (CIK 0001013857),8-K,2023-01-03,0001193125-23-000843:d442682dex991.htm,0001193125-23-000843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,PEGA
1,"""AI capabilities""",876167,PROGRESS SOFTWARE CORP /MA (PRGS) (CIK 00008...,8-K,2023-01-03,0000876167-23-000004:pressrelease-marklogic.htm,0000876167-23-000004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,PRGS
2,"""AI-powered""",1293818,OPGEN INC (OPGN) (CIK 0001293818),8-K,2023-01-04,0001079973-23-000010:ex99x1.htm,0001079973-23-000010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,CFOR
3,"""neural network""",278165,OMNIQ Corp. (OMQS) (CIK 0000278165),8-K,2023-01-04,0001493152-23-000229:ex99-1.htm,0001493152-23-000229,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,OMQS
4,"""deep learning""",1577445,ScoutCam Inc. (ODYS) (CIK 0001577445),8-K,2023-01-05,0001493152-23-000438:ex99-1.htm,0001493152-23-000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,ODYS


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/cik_ticker_map.csv"
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} commit -m "Step 3b: merge CIK-to-ticker mapping into screening worksheet"
!git -C {BASE_DIR} push

[main f504284] Step 3b: merge CIK-to-ticker mapping into screening worksheet
 2 files changed, 17032 insertions(+), 15142 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 136.52 KiB | 1.52 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   a6c9791..f504284  main -> main
